# SymPyによる量子力学：波動関数と不確定性原理の記号的検証

## 概要
量子力学は、微視的な世界を記述する物理学の体系であり、波動関数や演算子といった数学的道具を駆使する。これらの計算、特に波動関数の規格化や期待値の計算、あるいは交換関係の確認などは、手計算では非常に煩雑になることが多い。

Pythonの数式処理ライブラリ **SymPy** は、積分計算や行列演算に強く、量子力学の基礎的な問題を記号的に解くのに適している。さらに `sympy.physics.quantum` という量子力学専用のモジュールも存在し、ブラ・ケット記法などもサポートしている。

本記事では、SymPyを用いて以下のトピックを解説する。

1. **波動力学**：1次元無限井戸型ポテンシャルにおける波動関数の規格化と直交性
2. **不確定性原理**：位置と運動量の期待値・分散の計算と、ハイゼンベルクの不確定性原理の検証
3. **行列力学**：パウリ行列と交換関係

### 筆者の環境
筆者の実行環境は以下の通りである。

In [ ]:
!sw_vers

In [ ]:
!python -V

必要なライブラリをインポートする。虚数単位 `I` や円周率 `pi`、無限大 `oo` なども SymPy から取得する。

In [ ]:
import sympy
from sympy import symbols, sin, cos, sqrt, exp, integrate, diff, conjugate, simplify, pi, I, oo, Matrix
from sympy.physics.quantum import Dagger
from pprint import pprint as py_pprint

# 数式をLaTeX形式で綺麗に表示するための設定
sympy.init_printing()

print("sympy version :", sympy.__version__)

## 1. 1次元無限井戸型ポテンシャル

幅 $L$ の1次元無限井戸型ポテンシャル（$0 < x < L$ で $V(x)=0$、それ以外で $V(x)=\infty$）中の粒子を考える。
この系の固有関数（波動関数）は以下のように与えられる。

$$ \psi_n(x) = \sqrt{\frac{2}{L}} \sin\left(\frac{n \pi x}{L}\right) $$

ここで $n$ は正の整数である。

In [ ]:
x = symbols('x', real=True)
L = symbols('L', real=True, positive=True)
n = symbols('n', integer=True, positive=True)

psi = sqrt(2/L) * sin(n * pi * x / L)

print("波動関数 psi_n(x):")
sympy.pprint(psi)

### 1.1 規格化の確認

波動関数が規格化されているか、つまり全空間での存在確率が1になるかを確認してみる。

$$ \int_0^L |\psi_n(x)|^2 dx = 1 $$

In [ ]:
prob_density = abs(psi)**2
norm = integrate(prob_density, (x, 0, L))

print("規格化積分:")
sympy.pprint(norm)

結果は 1 となり、正しく規格化されていることが確認できた。

### 1.2 直交性の確認

異なる量子数 $n, m$ に対応する波動関数は直交するはずである。

$$ \int_0^L \psi_m^*(x) \psi_n(x) dx = \delta_{mn} $$

In [ ]:
m = symbols('m', integer=True, positive=True)
psi_m = psi.subs(n, m)

# n != m の場合を計算
ortho_check = integrate(psi_m * psi, (x, 0, L))

# n != m という条件を考慮して簡約化
# SymPyのsimplifyは整数の性質を自動的に使うが、明示的に確認する
print("直交性積分 (一般の n, m):")
sympy.pprint(simplify(ortho_check))

結果は 0 となり、直交性が確認できた（$n \neq m$ の場合）。

## 2. 不確定性原理の検証

ハイゼンベルクの不確定性原理によれば、位置の不確定性 $\sigma_x$ と運動量の不確定性 $\sigma_p$ の積には下限が存在する。

$$ \sigma_x \sigma_p \ge \frac{\hbar}{2} $$

これを基底状態 ($n=1$) について検証してみる。

### 2.1 位置の期待値と分散

位置演算子は $\hat{x} = x$ である。

$$ \langle x \rangle = \int_0^L \psi^* x \psi dx $$
$$ \langle x^2 \rangle = \int_0^L \psi^* x^2 \psi dx $$
$$ \sigma_x^2 = \langle x^2 \rangle - \langle x \rangle^2 $$

In [ ]:
psi_1 = psi.subs(n, 1)

exp_x = integrate(psi_1 * x * psi_1, (x, 0, L))
exp_x2 = integrate(psi_1 * x**2 * psi_1, (x, 0, L))

var_x = simplify(exp_x2 - exp_x**2)

print("位置の分散 sigma_x^2:")
sympy.pprint(var_x)

### 2.2 運動量の期待値と分散

運動量演算子は $\hat{p} = -i \hbar \frac{d}{dx}$ である。

$$ \langle p \rangle = \int_0^L \psi^* \left(-i \hbar \frac{d}{dx}\right) \psi dx $$
$$ \langle p^2 \rangle = \int_0^L \psi^* \left(-\hbar^2 \frac{d^2}{dx^2}\right) \psi dx $$

In [ ]:
hbar = symbols('hbar', real=True, positive=True)

# 運動量演算子の作用
p_op_psi = -I * hbar * diff(psi_1, x)
p2_op_psi = -hbar**2 * diff(psi_1, x, 2)

exp_p = integrate(psi_1 * p_op_psi, (x, 0, L))
exp_p2 = integrate(psi_1 * p2_op_psi, (x, 0, L))

var_p = simplify(exp_p2 - exp_p**2)

print("運動量の分散 sigma_p^2:")
sympy.pprint(var_p)

### 2.3 不確定性積の計算

$\sigma_x^2 \sigma_p^2$ を計算し、$\hbar^2/4$ と比較してみる。

In [ ]:
uncertainty_product = simplify(var_x * var_p)

print("不確定性積 sigma_x^2 * sigma_p^2:")
sympy.pprint(uncertainty_product)

# 数値的に評価して hbar^2/4 より大きいか確認
# pi^2 approx 9.87
# (L^2 * (pi^2 - 6) / (12 * pi^2)) * (pi^2 * hbar^2 / L^2)
# = hbar^2 * (pi^2 - 6) / 12
# (9.87 - 6) / 12 = 3.87 / 12 approx 0.32
# 一方 1/4 = 0.25 なので、0.32 > 0.25 で成立している

check = (uncertainty_product / hbar**2).evalf()
print("\n係数の数値評価 (hbar^2 の係数):", check)
print("1/4 の値:", 0.25)

係数は約 0.322 であり、0.25 ($1/4$) より大きいため、不確定性原理 $\sigma_x \sigma_p \ge \hbar/2$ （二乗すれば $\ge \hbar^2/4$）が成立していることが確認できた。

## 3. 行列力学：スピンとパウリ行列

電子のスピンなどを記述するパウリ行列 $\sigma_x, \sigma_y, \sigma_z$ は以下のように定義される。

$$ \sigma_x = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad \sigma_y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad \sigma_z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} $$

In [ ]:
sigma_x = Matrix([[0, 1], [1, 0]])
sigma_y = Matrix([[0, -I], [I, 0]])
sigma_z = Matrix([[1, 0], [0, -1]])

print("パウリ行列 sigma_x:")
sympy.pprint(sigma_x)

### 3.1 交換関係の確認

パウリ行列は以下の交換関係を満たすはずである。

$$ [\sigma_x, \sigma_y] = \sigma_x \sigma_y - \sigma_y \sigma_x = 2i \sigma_z $$

これをSymPyで計算してみる。

In [ ]:
comm_xy = sigma_x * sigma_y - sigma_y * sigma_x

print("交換子 [sigma_x, sigma_y]:")
sympy.pprint(comm_xy)

print("\n2i * sigma_z と一致するか:")
sympy.pprint(2 * I * sigma_z)
print(comm_xy == 2 * I * sigma_z)

一致することが確認できた。行列演算もSymPyを使えば定義通りに記述するだけで検証可能である。

## 結論

この記事では、SymPyを用いて量子力学の基礎的な問題を解析した。

波動関数の積分計算から、不確定性原理の厳密な検証、そして行列力学における交換関係の確認まで、SymPyは量子力学の学習や研究において強力な補助ツールとなる。特に、複雑な積分や行列計算のミスを防ぎ、物理的な本質の理解に集中できる点は大きなメリットである。

### 参考文献
- [SymPy Documentation: Quantum Mechanics](https://docs.sympy.org/latest/modules/physics/quantum/index.html)